# Notebook 01 — Exploratory Data Analysis

**Project:** Reducing Scrap in Injection Molding: A DAG-Informed Causal Decision Analysis  
**Dataset:** 5,000 half-hour production intervals | 12 machines | 4 plants | 33 columns

---

## Purpose

This notebook explores the dataset before any causal modelling. The goals are:

1. Understand the shape and quality of the data
2. Characterise the target KPI (`scrap_rate_pct`)
3. Break down defect contribution
4. Inspect raw correlations (noting that these are diagnostic, not causal)
5. Identify potential confounders and interaction patterns

The observations here motivate — but do not yet justify — the intervention recommendations. That work begins in Notebook 02.


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.utils import load_data, PROCESS_LEVERS, PLANNING_LEVER, CONFOUNDERS, MEDIATORS, PASS_FAIL_THRESHOLD
from src.plotting import (
    plot_scrap_distribution, plot_defect_breakdown,
    plot_correlation_heatmap, plot_warpage_interaction
)

pd.set_option("display.float_format", "{:.3f}".format)
print("Libraries loaded.")


## 2. Load and Inspect the Data

In [ ]:
df = load_data()
print(f"Shape: {df.shape}")
print(f"Date range: {df['timestamp'].min().date()}  →  {df['timestamp'].max().date()}")
print(f"Plants:  {sorted(df['plant_id'].unique())}")
print(f"Machines: {df['machine_id'].nunique()}")
print(f"Missing values: {df.isnull().sum().sum()}")


In [ ]:
df.head(3)


In [ ]:
df.describe().T.round(3)


## 3. Target KPI: `scrap_rate_pct`

The primary business metric. One entry per 30-minute production interval.

A pass/fail threshold of **3.2%** is used to classify intervals.


In [ ]:
pct_fail = (df["scrap_rate_pct"] > PASS_FAIL_THRESHOLD).mean() * 100
print(f"Mean scrap rate:      {df['scrap_rate_pct'].mean():.3f}%")
print(f"Median scrap rate:    {df['scrap_rate_pct'].median():.3f}%")
print(f"Std dev:              {df['scrap_rate_pct'].std():.3f} p.p.")
print(f"Min / Max:            {df['scrap_rate_pct'].min():.3f}% / {df['scrap_rate_pct'].max():.3f}%")
print(f"% intervals failing:  {pct_fail:.1f}%")


In [ ]:
fig = plot_scrap_distribution(df)
plt.show()


**Key observation:** The scrap rate distribution is unimodal and right-skewed. The bulk of intervals are already above the 3.2% threshold — this is a chronic process problem, not an outlier problem.


## 4. Defect Breakdown

Defect type is the dominant qualitative signal in the dataset. Each interval is classified by its primary defect mode.


In [ ]:
defect_stats = df.groupby("defect_type").agg(
    count=("scrap_rate_pct", "count"),
    mean_scrap=("scrap_rate_pct", "mean"),
    median_scrap=("scrap_rate_pct", "median"),
    share_pct=("scrap_rate_pct", lambda x: len(x)/len(df)*100),
).sort_values("count", ascending=False)
defect_stats.round(3)


In [ ]:
fig = plot_defect_breakdown(df)
plt.show()


**Key observations:**

- **Warpage** is the largest single defect (33% of intervals). It is the primary target for the cooling-time and mold-temperature recommendations.
- **Splay moisture** has the highest mean scrap per interval (4.74%) but is driven through the dryer pathway, which has limited statistical leverage (R² = 0.09 for the moisture sub-model).
- **Flash** and **dimensional deviation** together account for 37% of intervals and are sensitive to tool wear × pressure interactions.


## 5. Scrap by Plant and Machine

In [ ]:
plant_stats = df.groupby("plant_id")["scrap_rate_pct"].agg(["mean", "median", "std", "count"])
plant_stats = plant_stats.sort_values("mean", ascending=False)
plant_stats.columns = ["Mean scrap %", "Median scrap %", "Std dev", "N intervals"]
plant_stats.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
df.boxplot(column="scrap_rate_pct", by="plant_id", ax=ax,
           boxprops=dict(color="#264653"), medianprops=dict(color="#e76f51", lw=2),
           whiskerprops=dict(color="#264653"), capprops=dict(color="#264653"),
           flierprops=dict(marker=".", markersize=2, color="#adb5bd"))
ax.set_title("Scrap Rate by Plant")
ax.set_xlabel("Plant")
ax.set_ylabel("scrap_rate_pct (%)")
plt.suptitle("")
plt.tight_layout()
plt.show()


**Note:** `VN_QUANGNAM` has the highest mean scrap rate and also the highest ambient humidity exposure — consistent with the moisture pathway being more active there. This makes it the natural choice for the Phase 1 pilot.


## 6. Scrap Rate Over Time

In [ ]:
df_daily = df.set_index("timestamp").resample("D")["scrap_rate_pct"].mean()

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(df_daily.index, df_daily.values, color="#2a9d8f", lw=1.2, alpha=0.8)
ax.axhline(df["scrap_rate_pct"].mean(), color="#e76f51", lw=1.4, ls="--", label="Overall mean")
ax.axhline(3.2, color="#264653", lw=1, ls=":", label="Pass/fail threshold (3.2%)")
ax.set_title("Daily Mean Scrap Rate (all plants, all machines)")
ax.set_xlabel("Date")
ax.set_ylabel("Mean scrap_rate_pct (%)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## 7. Raw Correlations

**Important caveat:** Raw correlations are diagnostic. They identify co-variation between features and the target but they do not distinguish:
- forward causal effects from reverse causation
- direct effects from confounded proxies
- controllable levers from downstream symptoms

The cooling-time example (below) is the canonical illustration of why this matters.


In [ ]:
numeric_levers = [c for c in PROCESS_LEVERS + PLANNING_LEVER if c in df.columns]
corr_with_target = df[numeric_levers + ["scrap_rate_pct"]].corr()["scrap_rate_pct"].drop("scrap_rate_pct")
corr_with_target = corr_with_target.sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#e76f51" if v > 0 else "#2a9d8f" for v in corr_with_target.values]
corr_with_target.plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Raw Pearson Correlation with scrap_rate_pct\n(process levers + planning lever)")
ax.set_xlabel("ρ")
plt.tight_layout()
plt.show()

print("Raw correlations:")
print(corr_with_target.round(3))


**The cooling-time problem:**

`cooling_time_s` has a **raw correlation of ρ ≈ +0.28** with scrap, suggesting more cooling is associated with more scrap. This would motivate a recommendation to *shorten* cooling.

The causal analysis in Notebook 02 reverses this sign. Operators reactively extend cooling when they observe high mold temperatures (a source of thermal stress). The raw positive correlation captures this compensation behaviour — not a direct effect of cooling on scrap. After adjusting for mold temperature, the causal effect of cooling is **strongly protective (β = −1.75 std)**.


## 8. Pairwise Correlation Heatmap

In [ ]:
heatmap_cols = numeric_levers + ["ambient_humidity_pct", "ambient_temperature_c", "scrap_rate_pct"]
fig = plot_correlation_heatmap(df, heatmap_cols)
plt.show()


## 9. Warpage Interaction Preview

As a teaser for Notebook 02: within the warpage subset, mean scrap varies substantially by mold temperature × cooling time bin. The pattern points directly at the causal mechanism.


In [ ]:
fig = plot_warpage_interaction(df)
plt.show()


**Reading the heatmap:** At mold temperatures > 75 °C with cooling ≤ 12 s, mean scrap reaches **8.2%**. Extending cooling beyond 18 s at the same mold temperature brings this down to **5.1%** (−3.1 p.p.). At low mold temperatures, the cooling-time effect is negligible. This interaction is the strongest signal in the dataset.


## 10. EDA Summary

| Finding | Value |
|---|---|
| Mean scrap rate | 4.44% |
| % intervals failing 3.2% threshold | 78% |
| Dominant defect | Warpage (33%) |
| Defect with highest mean scrap | Splay moisture (4.74%) |
| Raw cooling-time correlation | ρ ≈ +0.28 (misleading) |
| Most correlated lever (raw) | `mold_temperature_c` (ρ ≈ +0.55) |
| Plant with highest scrap | VN_QUANGNAM |

**Next:** Notebook 02 formalises variable roles using the DAG, computes DAG-adjusted effect estimates, and demonstrates the cooling-time sign reversal.
